In [ ]:
# Step 1: Mount Google Drive
# (Skip this if running locally)
from google.colab import drive
drive.mount('/content/drive')

print('Drive mounted successfully ✅')

In [ ]:
# Step 2: Set the base path to where I uploaded my datasets
# Change this to your actual Drive folder path
BASE = '/content/drive/MyDrive/Ocean_Plastic_Project/data/raw/'

# Quick check — does the folder exist?
import os
if os.path.exists(BASE):
    print('Folder found ✅')
    print('Files inside:')
    for f in os.listdir(BASE):
        print(f'  {f}')
else:
    print('Folder not found ❌ — check your BASE path')

In [ ]:
# Step 3: Import only what I need for exploration right now
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('Libraries loaded ✅')

---
## 📍 PART 1: Define Our Bounding Box
This is the core geographic filter for the entire project. I'll define it once and reuse it everywhere.

In [ ]:
# Our North Indian Ocean bounding box
LAT_MIN = 5    # 5 degrees North
LAT_MAX = 30   # 30 degrees North
LON_MIN = 65   # 65 degrees East
LON_MAX = 95   # 95 degrees East

print('Bounding Box defined:')
print(f'  Latitude  : {LAT_MIN}°N  to  {LAT_MAX}°N')
print(f'  Longitude : {LON_MIN}°E  to  {LON_MAX}°E')
print()
print('This covers:')
print('  Arabian Sea | Bay of Bengal | Lakshadweep Sea')
print('  Gulf of Mannar | Andaman Sea | Coastal Indian Zones')

---
## 📍 PART 2: Explore ocean_plastic_pollution_data.csv
This is my **primary plastic dataset** (15,000 records). Let me check how many fall inside my bounding box.

In [ ]:
# Load the plastic dataset
df_plastic = pd.read_csv(BASE + 'ocean_plastic_pollution_data.csv')

print(f'Total records loaded: {len(df_plastic)}')
print(f'Columns: {df_plastic.columns.tolist()}')
print()
print('Lat range (global):', df_plastic['Latitude'].min().round(2), 'to', df_plastic['Latitude'].max().round(2))
print('Lon range (global):', df_plastic['Longitude'].min().round(2), 'to', df_plastic['Longitude'].max().round(2))
print()
print('Ocean regions in dataset:')
print(df_plastic['Region'].value_counts())

In [ ]:
# Apply bounding box filter
mask_plastic = (
    (df_plastic['Latitude']  >= LAT_MIN) &
    (df_plastic['Latitude']  <= LAT_MAX) &
    (df_plastic['Longitude'] >= LON_MIN) &
    (df_plastic['Longitude'] <= LON_MAX)
)

df_plastic_box = df_plastic[mask_plastic].copy()

print(f'Records inside bounding box: {len(df_plastic_box)} / {len(df_plastic)}')
print(f'({round(len(df_plastic_box)/len(df_plastic)*100, 1)}% of total)')
print()
print('Regions found inside box:')
print(df_plastic_box['Region'].value_counts())
print()
print('NOTE: Even records labelled Arctic/Atlantic/Southern appear here')
print('because Region is just a label in the data, not derived from coordinates.')
print('We will rely on coordinates, not Region label, for spatial accuracy.')

In [ ]:
# What geographic zones do these coordinates correspond to?
# Let me manually map some coordinate ranges to known ocean zones

def identify_zone(lat, lon):
    """Simple rule-based zone identifier for the Indian Ocean bounding box."""
    if 5 <= lat <= 25 and 65 <= lon <= 75:
        return 'Arabian Sea'
    elif 5 <= lat <= 22 and 75 <= lon <= 90:
        return 'Bay of Bengal'
    elif 10 <= lat <= 25 and 90 <= lon <= 95:
        return 'Andaman Sea'
    elif 5 <= lat <= 12 and 72 <= lon <= 80:
        return 'Lakshadweep / Gulf of Mannar'
    elif 20 <= lat <= 30 and 65 <= lon <= 75:
        return 'Gulf of Kutch / Rajasthan Coast'
    elif 22 <= lat <= 30 and 75 <= lon <= 95:
        return 'Northern India / Himalayan Foothills'
    else:
        return 'Indian Ocean (general)'

df_plastic_box['Zone'] = df_plastic_box.apply(
    lambda r: identify_zone(r['Latitude'], r['Longitude']), axis=1
)

print('Geographic zone distribution of plastic records:')
print(df_plastic_box['Zone'].value_counts())

In [ ]:
# Visualise plastic record locations inside the bounding box
fig, ax = plt.subplots(figsize=(10, 7))

# Color by zone
zone_colors = {
    'Arabian Sea': 'royalblue',
    'Bay of Bengal': 'darkorange',
    'Andaman Sea': 'green',
    'Lakshadweep / Gulf of Mannar': 'purple',
    'Gulf of Kutch / Rajasthan Coast': 'red',
    'Northern India / Himalayan Foothills': 'brown',
    'Indian Ocean (general)': 'gray'
}

for zone, color in zone_colors.items():
    subset = df_plastic_box[df_plastic_box['Zone'] == zone]
    ax.scatter(subset['Longitude'], subset['Latitude'],
               c=color, label=zone, alpha=0.6, s=20)

# Draw bounding box
rect = mpatches.Rectangle(
    (LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
    linewidth=2, edgecolor='red', facecolor='none', linestyle='--', label='Bounding Box'
)
ax.add_patch(rect)

# Labels for major cities/zones
city_labels = {
    'Mumbai': (19.0, 72.8),
    'Chennai': (13.0, 80.3),
    'Kolkata': (22.6, 88.4),
    'Kochi': (9.9, 76.3),
    'Visakhapatnam': (17.7, 83.3),
    'Andaman Islands': (12.0, 92.7),
}
for city, (lat, lon) in city_labels.items():
    ax.annotate(city, (lon, lat), fontsize=7, color='black',
                ha='center',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6))

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Plastic Pollution Records — Inside Bounding Box\n(Arabian Sea, Bay of Bengal & Indian Ocean)', fontsize=13)
ax.legend(loc='upper left', fontsize=7)
ax.grid(True, alpha=0.3)
ax.set_xlim(62, 98)
ax.set_ylim(3, 32)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Ocean_Plastic_Project/visualizations/geo_plastic_bbox.png', dpi=150)
plt.show()
print('Plot saved ✅')

---
## 📍 PART 3: Explore india_cms_final_master_enriched.csv
My **primary marine species dataset** (12,374 records). Let me see which states and coastal zones are covered.

In [ ]:
df_species = pd.read_csv(BASE + 'india_cms_final_master_enriched.csv')

print(f'Total records: {len(df_species)}')
print(f'Columns: {df_species.columns.tolist()}')
print()
print('Lat range:', df_species['latitude'].min(), 'to', df_species['latitude'].max())
print('Lon range:', df_species['longitude'].min(), 'to', df_species['longitude'].max())
print()
# NOTE: lat goes to 60 and year has -980 anomalies — flag these
print('⚠️  Lat goes up to 60 (outside India) — anomaly spotted')
print('⚠️  Year range:', df_species['year'].min(), 'to', df_species['year'].max(), '— clear anomalies')

In [ ]:
# Apply bounding box filter
mask_species = (
    (df_species['latitude']  >= LAT_MIN) &
    (df_species['latitude']  <= LAT_MAX) &
    (df_species['longitude'] >= LON_MIN) &
    (df_species['longitude'] <= LON_MAX)
)

df_species_box = df_species[mask_species].copy()

print(f'Records inside bounding box: {len(df_species_box)} / {len(df_species)}')
print()
print('States/Provinces covered:')
print(df_species_box['state_province'].value_counts().head(10))
print()
print('Taxonomic groups:')
print(df_species_box['taxonomic_group'].value_counts())
print()
print('Top 10 species:')
print(df_species_box['species_common_name'].value_counts().head(10))

In [ ]:
# Plot species observation locations
fig, ax = plt.subplots(figsize=(10, 7))

group_colors = {
    'Aves': 'steelblue',
    'Mammalia': 'darkorange',
    'Reptilia': 'green',
    'Pisces': 'purple'
}

for group, color in group_colors.items():
    subset = df_species_box[df_species_box['taxonomic_group'] == group]
    ax.scatter(subset['longitude'], subset['latitude'],
               c=color, label=f'{group} ({len(subset)})', alpha=0.3, s=8)

# Bounding box
rect = mpatches.Rectangle(
    (LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
    linewidth=2, edgecolor='red', facecolor='none', linestyle='--'
)
ax.add_patch(rect)

# City labels
for city, (lat, lon) in city_labels.items():
    ax.annotate(city, (lon, lat), fontsize=7, color='black',
                ha='center',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Marine Species Observations — Inside Bounding Box\nby Taxonomic Group', fontsize=13)
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(62, 98)
ax.set_ylim(3, 32)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Ocean_Plastic_Project/visualizations/geo_species_bbox.png', dpi=150)
plt.show()
print('Plot saved ✅')

---
## 📍 PART 4: Overlay — Plastic + Species on Same Map
This is the key pre-check: **do plastic records and species records actually overlap geographically?**
If they don't, the spatial join in Week 3 will produce an empty master table.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

# Plot species (background, small dots)
ax.scatter(
    df_species_box['longitude'], df_species_box['latitude'],
    c='steelblue', alpha=0.15, s=6, label=f'Species ({len(df_species_box)} obs)'
)

# Plot plastic (foreground, bigger dots)
ax.scatter(
    df_plastic_box['Longitude'], df_plastic_box['Latitude'],
    c='red', alpha=0.7, s=40, marker='x', label=f'Plastic ({len(df_plastic_box)} records)', zorder=5
)

# Bounding box
rect = mpatches.Rectangle(
    (LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
    linewidth=2.5, edgecolor='darkred', facecolor='none', linestyle='--'
)
ax.add_patch(rect)

# City labels
for city, (lat, lon) in city_labels.items():
    ax.annotate(city, (lon, lat), fontsize=7.5, color='black',
                ha='center',
                bbox=dict(boxstyle='round,pad=0.2', fc='lightyellow', alpha=0.9))

# Ocean zone labels
ax.text(70, 16, 'Arabian Sea', fontsize=9, color='navy', style='italic', alpha=0.7)
ax.text(85, 14, 'Bay of Bengal', fontsize=9, color='navy', style='italic', alpha=0.7)
ax.text(91, 11, 'Andaman\nSea', fontsize=8, color='navy', style='italic', alpha=0.7)

ax.set_xlabel('Longitude (°E)', fontsize=11)
ax.set_ylabel('Latitude (°N)', fontsize=11)
ax.set_title('Geographic Overlap Check\nPlastic Records vs Species Observations in Bounding Box', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(62, 98)
ax.set_ylim(3, 32)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Ocean_Plastic_Project/visualizations/geo_overlap_check.png', dpi=150)
plt.show()
print('Overlap map saved ✅')

---
## 📋 PART 5: Geographic Summary — What We Found
Before moving forward, documenting what the coordinates actually represent.

In [ ]:
print('=' * 60)
print('GEOGRAPHIC EXPLORATION SUMMARY')
print('=' * 60)
print()
print('BOUNDING BOX: Lat 5°N–30°N | Lon 65°E–95°E')
print()
print('OCEAN ZONES COVERED:')
print('  Arabian Sea         — west coast of India (Mumbai, Goa, Kochi)')
print('  Bay of Bengal       — east coast (Chennai, Visakhapatnam, Kolkata)')
print('  Andaman Sea         — Andaman & Nicobar Islands (93°E)')
print('  Lakshadweep Sea     — island chain west of Kerala')
print('  Gulf of Mannar      — between Tamil Nadu and Sri Lanka')
print('  Gulf of Kutch       — Gujarat, northwest India')
print()
print('PLASTIC DATA:')
print(f'  Records in box: {len(df_plastic_box)} / {len(df_plastic)}')
print('  ⚠️  Region labels in data do NOT match coordinates')
print('     → Will use coordinates only for spatial join')
print()
print('SPECIES DATA:')
print(f'  Records in box: {len(df_species_box)} / {len(df_species)}')
print(f'  Top state: Coastal India / Marine ({df_species_box["state_province"].value_counts().iloc[0]} records)')
print('  ⚠️  Year column has extreme anomalies — fix in Week 2')
print()
print('OVERLAP CHECK:')
print('  Plastic and species records DO overlap geographically ✅')
print('  Spatial join in Week 3 should work correctly')
print()
print('READY TO PROCEED TO: May 19 — Environment Setup')
print('=' * 60)